In [ ]:
import os
import sys
from pathlib import Path

if sys.platform == 'linux':
    os.environ.setdefault('MUJOCO_GL', 'egl')
    os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')


# Shared Door + Pick-and-Place Demo

This notebook builds one simple MuJoCo scene with a hinged door, one pickup object, and a shelf target. It then renders two qualitative task demonstrations in the same scene geometry:

- `open the door`
- `put the mug on the shelf`

The rollouts are notebook-local scripted demonstrations. They are intended as a lightweight multi-task scene demo, not as a benchmark or a real policy evaluation.


## Bootstrap

Recommended environment setup from `MolmoBot/`:

```bash
uv sync --extra eval
sudo apt-get update
sudo apt-get install -y libegl1 libgl1 libgles2 libglfw3 libosmesa6 libgl1-mesa-dri mesa-utils
```

On Linux, the first code cell forces MuJoCo to use EGL for headless rendering.


In [ ]:
import math

import mujoco
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from IPython.display import Markdown, Video, display


In [ ]:
def renderer_smoke_test():
    xml = '''
    <mujoco model="smoke">
      <worldbody>
        <geom type="plane" size="1 1 0.1"/>
        <camera name="cam" pos="1 -1 1" xyaxes="0.707 0.707 0 -0.408 0.408 0.816"/>
      </worldbody>
    </mujoco>
    '''
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, 120, 160)
    renderer.update_scene(data, camera='cam')
    img = renderer.render()
    renderer.close()
    return img.shape

renderer_smoke_test()


In [ ]:
OUTPUT_DIR = Path('demo_outputs/door_pick_place_shared_scene')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RENDER_HEIGHT = 360
RENDER_WIDTH = 640
VIDEO_FPS = 20

SCENE_SPEC = {
    'object_name': 'mug',
    'placement_name': 'shelf',
    'door_open_angle': 1.15,
    'object_start_pos': [-0.35, -0.22, 0.06],
    'shelf_place_pos': [0.85, 0.02, 0.89],
    'gripper_home': [-0.55, -0.45, 0.62],
    'staging_pos': [0.12, -0.16, 0.68],
    'arm_base_pos': [-0.66, -0.36, 0.28],
    'arm_upper_len': 0.43,
    'arm_fore_len': 0.39,
    'door_grasp_offset': [-0.03, -0.005, 0.0],
    'door_retreat_pos': [-0.34, -0.10, 0.84],
}

TASK_SPECS = [
    {
        'task_id': 'door_open',
        'prompt': 'open the door',
        'duration_s': 4.5,
        'manual_note': 'review_pending',
    },
    {
        'task_id': 'pick_and_place',
        'prompt': 'put the mug on the shelf',
        'duration_s': 7.0,
        'manual_note': 'review_pending',
    },
]


In [ ]:
def make_demo_xml(scene_spec):
    object_xyz = ' '.join(str(v) for v in scene_spec['object_start_pos'])
    gripper_xyz = ' '.join(str(v) for v in scene_spec['gripper_home'])
    arm_base_xyz = ' '.join(str(v) for v in scene_spec['arm_base_pos'])
    upper_half = scene_spec['arm_upper_len'] / 2.0
    fore_half = scene_spec['arm_fore_len'] / 2.0
    return f"""
<mujoco model="shared_demo_scene">
  <option timestep="0.02"/>
  <visual>
    <global offwidth="{RENDER_WIDTH}" offheight="{RENDER_HEIGHT}"/>
    <headlight diffuse="0.8 0.8 0.8" ambient="0.35 0.35 0.35" specular="0.15 0.15 0.15"/>
  </visual>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.97 0.98 1.0" rgb2="0.77 0.84 0.93" width="256" height="256"/>
    <material name="floor" rgba="0.88 0.87 0.82 1"/>
    <material name="wall" rgba="0.95 0.95 0.97 1"/>
    <material name="door" rgba="0.58 0.38 0.20 1"/>
    <material name="shelf" rgba="0.67 0.69 0.73 1"/>
    <material name="object" rgba="0.14 0.42 0.80 1"/>
    <material name="gripper" rgba="0.15 0.15 0.18 1"/>
    <material name="arm" rgba="0.27 0.31 0.36 1"/>
    <material name="arm_joint" rgba="0.91 0.44 0.22 1"/>
  </asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" diffuse="1 1 1"/>
    <geom name="floor" type="plane" size="3 3 0.1" material="floor"/>
    <camera name="overview" pos="1.95 -1.65 1.40" xyaxes="0.74 0.67 0 -0.31 0.34 0.89"/>
    <camera name="side" pos="-0.65 -1.35 1.10" xyaxes="0.98 -0.18 0 0.12 0.66 0.74"/>

    <body name="cabinet_frame" pos="0.55 0 0">
      <geom type="box" pos="0 0 0.75" size="0.03 0.50 0.75" material="wall"/>
      <geom type="box" pos="0.31 0 1.47" size="0.34 0.50 0.03" material="wall"/>
      <geom type="box" pos="0.31 0 0.03" size="0.34 0.50 0.03" material="wall"/>
      <geom type="box" pos="0.63 0 0.75" size="0.03 0.50 0.75" material="wall"/>
      <geom type="box" pos="0.31 0 0.82" size="0.28 0.22 0.025" material="shelf"/>

      <body name="door" pos="0.03 -0.50 0.75">
        <joint name="door_hinge" type="hinge" axis="0 0 1" range="0 1.45" damping="0.2"/>
        <geom type="box" pos="0.27 0.02 0" size="0.27 0.02 0.68" material="door"/>
        <geom name="door_handle" type="capsule" fromto="0.43 0.035 -0.04 0.49 0.035 0.04" size="0.018" rgba="0.95 0.82 0.32 1"/>
        <site name="handle_site" pos="0.47 0.035 0" size="0.016" rgba="1 0.2 0.2 1"/>
      </body>
    </body>

    <body name="pickup_object" pos="{object_xyz}">
      <freejoint name="object_free"/>
      <geom type="box" size="0.045 0.045 0.045" material="object"/>
      <site name="object_site" pos="0 0 0" size="0.015" rgba="0.15 0.55 1 1"/>
    </body>

    <body name="arm_base" pos="{arm_base_xyz}">
      <geom type="cylinder" size="0.06 0.18" material="arm_joint"/>
      <geom type="box" pos="0.00 0.00 0.21" size="0.07 0.07 0.03" material="arm"/>
      <site name="arm_mount_site" pos="0.00 0.00 0.18" size="0.01" rgba="1 1 1 1"/>
    </body>

    <body name="upper_arm_visual" mocap="true" pos="-0.45 -0.30 0.45">
      <geom type="capsule" pos="0 0 {upper_half}" size="0.028 {upper_half}" material="arm" contype="0" conaffinity="0"/>
    </body>

    <body name="forearm_visual" mocap="true" pos="-0.20 -0.18 0.65">
      <geom type="capsule" pos="0 0 {fore_half}" size="0.024 {fore_half}" material="arm" contype="0" conaffinity="0"/>
    </body>

    <body name="elbow_visual" mocap="true" pos="-0.33 -0.24 0.56">
      <geom type="sphere" size="0.035" material="arm_joint" contype="0" conaffinity="0"/>
    </body>

    <body name="gripper" mocap="true" pos="{gripper_xyz}">
      <geom type="box" size="0.05 0.035 0.025" material="gripper" contype="0" conaffinity="0"/>
      <geom type="box" pos="0.06 0 0.04" size="0.012 0.012 0.04" material="gripper" contype="0" conaffinity="0"/>
      <geom type="box" pos="0.06 0 -0.04" size="0.012 0.012 0.04" material="gripper" contype="0" conaffinity="0"/>
      <site name="grip_site" pos="0.08 0 0" size="0.012" rgba="0.2 1 0.2 1"/>
    </body>
  </worldbody>
</mujoco>
"""


def build_scene(scene_spec=SCENE_SPEC):
    model = mujoco.MjModel.from_xml_string(make_demo_xml(scene_spec))
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, RENDER_HEIGHT, RENDER_WIDTH)

    object_joint_id = model.joint('object_free').id
    object_qpos_adr = model.jnt_qposadr[object_joint_id]
    door_joint_id = model.joint('door_hinge').id
    door_qpos_adr = model.jnt_qposadr[door_joint_id]
    mocap_body_names = ['upper_arm_visual', 'forearm_visual', 'elbow_visual', 'gripper']
    mocap_body_ids = {name: model.body(name).mocapid[0] for name in mocap_body_names}

    scene_ctx = {
        'model': model,
        'data': data,
        'renderer': renderer,
        'scene_spec': scene_spec,
        'object_qpos_adr': object_qpos_adr,
        'door_qpos_adr': door_qpos_adr,
        'mocap_body_ids': mocap_body_ids,
        'camera_names': ['overview', 'side'],
    }
    reset_scene(scene_ctx)
    return scene_ctx


def close_scene(scene_ctx):
    scene_ctx['renderer'].close()


def set_free_body_pose(scene_ctx, pos, quat=(1.0, 0.0, 0.0, 0.0)):
    adr = scene_ctx['object_qpos_adr']
    scene_ctx['data'].qpos[adr:adr + 3] = np.asarray(pos, dtype=float)
    scene_ctx['data'].qpos[adr + 3:adr + 7] = np.asarray(quat, dtype=float)


def normalize(vec):
    vec = np.asarray(vec, dtype=float)
    norm = np.linalg.norm(vec)
    if norm < 1e-8:
        return np.array([0.0, 0.0, 1.0])
    return vec / norm


def quat_from_matrix(rot):
    m = np.asarray(rot, dtype=float)
    trace = np.trace(m)
    if trace > 0.0:
        s = math.sqrt(trace + 1.0) * 2.0
        return np.array([
            0.25 * s,
            (m[2, 1] - m[1, 2]) / s,
            (m[0, 2] - m[2, 0]) / s,
            (m[1, 0] - m[0, 1]) / s,
        ])
    idx = int(np.argmax(np.diag(m)))
    if idx == 0:
        s = math.sqrt(1.0 + m[0, 0] - m[1, 1] - m[2, 2]) * 2.0
        return np.array([
            (m[2, 1] - m[1, 2]) / s,
            0.25 * s,
            (m[0, 1] + m[1, 0]) / s,
            (m[0, 2] + m[2, 0]) / s,
        ])
    if idx == 1:
        s = math.sqrt(1.0 + m[1, 1] - m[0, 0] - m[2, 2]) * 2.0
        return np.array([
            (m[0, 2] - m[2, 0]) / s,
            (m[0, 1] + m[1, 0]) / s,
            0.25 * s,
            (m[1, 2] + m[2, 1]) / s,
        ])
    s = math.sqrt(1.0 + m[2, 2] - m[0, 0] - m[1, 1]) * 2.0
    return np.array([
        (m[1, 0] - m[0, 1]) / s,
        (m[0, 2] + m[2, 0]) / s,
        (m[1, 2] + m[2, 1]) / s,
        0.25 * s,
    ])


def segment_quat(start, end):
    direction = normalize(np.asarray(end) - np.asarray(start))
    up_hint = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(direction, up_hint)) > 0.92:
        up_hint = np.array([0.0, 1.0, 0.0])
    x_axis = normalize(np.cross(up_hint, direction))
    y_axis = normalize(np.cross(direction, x_axis))
    rot = np.column_stack([x_axis, y_axis, direction])
    quat = quat_from_matrix(rot)
    return quat / np.linalg.norm(quat)


def set_mocap_pose(scene_ctx, body_name, pos, quat=(1.0, 0.0, 0.0, 0.0)):
    mocap_id = scene_ctx['mocap_body_ids'][body_name]
    scene_ctx['data'].mocap_pos[mocap_id] = np.asarray(pos, dtype=float)
    scene_ctx['data'].mocap_quat[mocap_id] = np.asarray(quat, dtype=float)


def compute_elbow_point(base, grip, upper_len, fore_len):
    base = np.asarray(base, dtype=float)
    grip = np.asarray(grip, dtype=float)
    shoulder = base + np.array([0.0, 0.0, 0.18])
    reach = grip - shoulder
    dist = np.linalg.norm(reach)
    dist = np.clip(dist, 1e-6, upper_len + fore_len - 1e-6)
    direction = reach / dist
    bend_hint = np.array([0.22, 0.08, 0.34])
    lateral = bend_hint - direction * np.dot(bend_hint, direction)
    if np.linalg.norm(lateral) < 1e-6:
        lateral = np.array([0.0, 1.0, 0.0])
    lateral = lateral / np.linalg.norm(lateral)
    proj = (upper_len ** 2 - fore_len ** 2 + dist ** 2) / (2.0 * dist)
    height_sq = max(upper_len ** 2 - proj ** 2, 0.0)
    height = math.sqrt(height_sq)
    elbow = shoulder + direction * proj + lateral * height
    return shoulder, elbow


def update_arm_visuals(scene_ctx, grip_pos):
    spec = scene_ctx['scene_spec']
    base = np.asarray(spec['arm_base_pos'], dtype=float)
    shoulder, elbow = compute_elbow_point(base, grip_pos, spec['arm_upper_len'], spec['arm_fore_len'])
    upper_mid = 0.5 * (shoulder + elbow)
    fore_mid = 0.5 * (elbow + grip_pos)
    set_mocap_pose(scene_ctx, 'upper_arm_visual', upper_mid, segment_quat(shoulder, elbow))
    set_mocap_pose(scene_ctx, 'forearm_visual', fore_mid, segment_quat(elbow, grip_pos))
    set_mocap_pose(scene_ctx, 'elbow_visual', elbow)


def set_gripper_pose(scene_ctx, pos, quat=(1.0, 0.0, 0.0, 0.0)):
    pos = np.asarray(pos, dtype=float)
    set_mocap_pose(scene_ctx, 'gripper', pos, quat)
    update_arm_visuals(scene_ctx, pos)


def set_door_angle(scene_ctx, angle):
    scene_ctx['data'].qpos[scene_ctx['door_qpos_adr']] = angle


def refresh_scene(scene_ctx):
    mujoco.mj_forward(scene_ctx['model'], scene_ctx['data'])


def reset_scene(scene_ctx):
    set_door_angle(scene_ctx, 0.0)
    set_free_body_pose(scene_ctx, scene_ctx['scene_spec']['object_start_pos'])
    set_gripper_pose(scene_ctx, scene_ctx['scene_spec']['gripper_home'])
    refresh_scene(scene_ctx)


def site_pos(scene_ctx, site_name):
    site_id = scene_ctx['model'].site(site_name).id
    return scene_ctx['data'].site_xpos[site_id].copy()


def render_scene(scene_ctx, camera_names=None):
    if camera_names is None:
        camera_names = scene_ctx['camera_names']
    images = {}
    for camera_name in camera_names:
        scene_ctx['renderer'].update_scene(scene_ctx['data'], camera=camera_name)
        images[camera_name] = scene_ctx['renderer'].render().copy()
    return images


def preview_scene(scene_spec=SCENE_SPEC):
    scene_ctx = build_scene(scene_spec)
    images = render_scene(scene_ctx)
    close_scene(scene_ctx)
    return Image.fromarray(np.hstack([images['overview'], images['side']]))


In [ ]:
def lerp(a, b, alpha):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return (1.0 - alpha) * a + alpha * b


def smoothstep(alpha):
    alpha = np.clip(alpha, 0.0, 1.0)
    return alpha * alpha * (3.0 - 2.0 * alpha)


def phase_alpha(phase, start, end):
    if phase <= start:
        return 0.0
    if phase >= end:
        return 1.0
    return smoothstep((phase - start) / (end - start))


def door_pull_pose(scene_ctx, angle):
    spec = scene_ctx['scene_spec']
    set_door_angle(scene_ctx, angle)
    refresh_scene(scene_ctx)
    handle = site_pos(scene_ctx, 'handle_site')
    return handle + np.asarray(spec['door_grasp_offset'], dtype=float)


def apply_task_state(scene_ctx, task_spec, phase):
    spec = scene_ctx['scene_spec']
    home = np.asarray(spec['gripper_home'], dtype=float)
    staging = np.asarray(spec['staging_pos'], dtype=float)
    object_start = np.asarray(spec['object_start_pos'], dtype=float)
    shelf_place = np.asarray(spec['shelf_place_pos'], dtype=float)
    retreat = np.asarray(spec['door_retreat_pos'], dtype=float)
    closed_grasp = door_pull_pose(scene_ctx, 0.0)
    approach = closed_grasp + np.array([-0.10, -0.04, 0.05])
    task_id = task_spec['task_id']

    if task_id == 'door_open':
        if phase < 0.24:
            set_door_angle(scene_ctx, 0.0)
            gripper = lerp(home, approach, phase_alpha(phase, 0.0, 0.24))
        elif phase < 0.34:
            set_door_angle(scene_ctx, 0.0)
            gripper = lerp(approach, closed_grasp, phase_alpha(phase, 0.24, 0.34))
        elif phase < 0.72:
            pull_alpha = phase_alpha(phase, 0.34, 0.72)
            gripper = door_pull_pose(scene_ctx, spec['door_open_angle'] * pull_alpha)
        elif phase < 0.84:
            gripper = door_pull_pose(scene_ctx, spec['door_open_angle'])
        else:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(door_pull_pose(scene_ctx, spec['door_open_angle']), retreat, phase_alpha(phase, 0.84, 1.0))

        set_gripper_pose(scene_ctx, gripper)
        set_free_body_pose(scene_ctx, object_start)
        refresh_scene(scene_ctx)
        return

    if task_id == 'pick_and_place':
        pre_grasp = object_start + np.array([0.00, 0.00, 0.18])
        grasp = object_start + np.array([0.00, 0.00, 0.07])
        carry = shelf_place + np.array([-0.10, -0.10, 0.17])
        release = shelf_place + np.array([0.00, 0.00, 0.08])
        post_place = shelf_place + np.array([-0.20, -0.18, 0.14])

        if phase < 0.08:
            set_door_angle(scene_ctx, 0.0)
            gripper = home
            object_pos = object_start
        elif phase < 0.18:
            set_door_angle(scene_ctx, 0.0)
            gripper = lerp(home, approach, phase_alpha(phase, 0.08, 0.18))
            object_pos = object_start
        elif phase < 0.30:
            pull_alpha = phase_alpha(phase, 0.18, 0.30)
            gripper = door_pull_pose(scene_ctx, spec['door_open_angle'] * pull_alpha)
            object_pos = object_start
        elif phase < 0.38:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(door_pull_pose(scene_ctx, spec['door_open_angle']), staging, phase_alpha(phase, 0.30, 0.38))
            object_pos = object_start
        elif phase < 0.50:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(staging, pre_grasp, phase_alpha(phase, 0.38, 0.50))
            object_pos = object_start
        elif phase < 0.58:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(pre_grasp, grasp, phase_alpha(phase, 0.50, 0.58))
            object_pos = object_start
        elif phase < 0.76:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(grasp, carry, phase_alpha(phase, 0.58, 0.76))
            object_pos = gripper + np.array([0.00, 0.00, -0.07])
        elif phase < 0.88:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(carry, release, phase_alpha(phase, 0.76, 0.88))
            object_pos = gripper + np.array([0.00, 0.00, -0.07])
        else:
            set_door_angle(scene_ctx, spec['door_open_angle'])
            gripper = lerp(release, post_place, phase_alpha(phase, 0.88, 1.0))
            object_pos = shelf_place

        set_gripper_pose(scene_ctx, gripper)
        set_free_body_pose(scene_ctx, object_pos)
        refresh_scene(scene_ctx)
        return

    raise ValueError(f'Unknown task_id: {task_id}')


## Preview The Shared Scene


In [ ]:
preview_scene()


## Run The Demo Tasks

Each task starts from the same initial scene. The pick-and-place sequence opens the door as part of reaching the shelf target, so both demonstrations remain grounded in the same setup.


In [ ]:
def run_rollout(task_spec, scene_spec=SCENE_SPEC, output_dir=OUTPUT_DIR, fps=VIDEO_FPS):
    scene_ctx = build_scene(scene_spec)
    n_steps = max(2, round(task_spec['duration_s'] * fps))
    frames = []

    for step in tqdm(range(n_steps), desc=task_spec['task_id']):
        phase = step / (n_steps - 1)
        apply_task_state(scene_ctx, task_spec, phase)
        images = render_scene(scene_ctx)
        frames.append(np.hstack([images['overview'], images['side']]))

    video_path = output_dir / f"{task_spec['task_id']}.mp4"
    clip = ImageSequenceClip([frame for frame in frames], fps=fps)
    clip.write_videofile(str(video_path), audio=False, logger=None)
    clip.close()
    close_scene(scene_ctx)

    return {
        'task_id': task_spec['task_id'],
        'prompt': task_spec['prompt'],
        'scene': 'shared_door_pick_place_scene',
        'video_path': str(video_path),
        'manual_note': task_spec['manual_note'],
    }


def run_task_suite(task_specs=TASK_SPECS):
    results = [run_rollout(task_spec) for task_spec in task_specs]
    return pd.DataFrame(results)


results_df = run_task_suite()
results_df


In [ ]:
for record in results_df.to_dict(orient='records'):
    summary = (
        f"### {record['task_id']}\n"
        f"- prompt: `{record['prompt']}`\n"
        f"- scene: `{record['scene']}`\n"
        f"- note: `{record['manual_note']}`"
    )
    display(Markdown(summary))
    display(Video(record['video_path'], embed=True))
